# Task 2: PostgreSQL SQL and Python data extraction

This notebook connects SQLAlchemy/Pandas to PostgreSQL, loads the cleaned Telco customer snapshot, and runs SQL queries into DataFrames. Install dependencies with `pip install -r requirements.txt`, create the `apexplanet` PostgreSQL database, and copy `.env.example` to `.env` with your local credentials. Never commit `.env`.

The practice set (22 SELECT queries, plus a reusable view and EXPLAIN example) is in [`sql/task2_queries.sql`](../sql/task2_queries.sql). Run its statements in pgAdmin or `psql`. Monthly sales trends and product sales are not computable from this snapshot because it has no dates or order-line/product-sales data.

In [ ]:
import sys
from pathlib import Path

from sqlalchemy import text

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "postgres_db_utils.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find scripts/postgres_db_utils.py.")
sys.path.insert(0, str(project_root / "scripts"))

from postgres_db_utils import (
    create_postgres_engine,
    load_csv_to_postgres,
    read_sql_dataframe,
)

csv_path = project_root / "data" / "processed" / "telco_churn_clean.csv"
engine = create_postgres_engine()

In [ ]:
with engine.connect() as connection:
    postgres_version = connection.execute(text("SELECT VERSION()")).scalar_one()
print(f"Connected to PostgreSQL: {postgres_version}")

## Load the cleaned CSV

The default `fail` mode avoids silently replacing an existing table. Use `replace` only when you intentionally want to recreate the table.

In [ ]:
rows_loaded = load_csv_to_postgres(
    csv_path,
    engine,
    table_name="telco_churn",
    if_exists="fail",
)
print(f"Loaded {rows_loaded:,} rows into telco_churn")

## SQL + Python extraction

SQLAlchemy binds parameter values safely; Pandas returns the SQL result as a DataFrame for further Python analysis.

In [ ]:
query = """
SELECT contract,
       COUNT(*) AS customer_count,
       SUM(total_charges) AS lifetime_charges,
       ROUND(AVG(monthly_charges)::numeric, 2) AS avg_monthly_charges,
       ROUND(
           100.0 * COUNT(*) FILTER (WHERE churn = 'No') / COUNT(*),
           2
       ) AS retention_rate_pct
FROM telco_churn
WHERE tenure >= :minimum_tenure
GROUP BY contract
HAVING COUNT(*) >= :minimum_customers
ORDER BY lifetime_charges DESC
LIMIT :row_limit
"""
contract_summary = read_sql_dataframe(
    engine,
    query,
    params={
        "minimum_tenure": 12,
        "minimum_customers": 100,
        "row_limit": 10,
    },
)
contract_summary

In [ ]:
service_query = """
SELECT internet_service AS service_category,
       COUNT(*) AS customer_count,
       ROUND(AVG(monthly_charges)::numeric, 2) AS avg_monthly_charges,
       ROUND(
           100.0 * COUNT(*) FILTER (WHERE churn = 'Yes') / COUNT(*),
           2
       ) AS churn_rate_pct
FROM telco_churn
GROUP BY internet_service
ORDER BY churn_rate_pct DESC
"""
service_performance = read_sql_dataframe(engine, service_query)
service_performance

In [ ]:
engine.dispose()